# Statystyka zdarzeń

Zbiór do analizy: baza `2026-10-18 obiekt`, sesja 9, obecny wariant reguł sygnałów (`CURRENT_VARIANT`), tylko zdarzenia przyjęte. Plik rozpakowuje się z kopii w repo:

```shell
xz -dkc data/snapshots/prepared-obiekt-sesja-9.db.xz > data/prepared/obiekt-sesja-9.db
```

Dla każdego zdarzenia (`dms_analysis.event_stats`) różnica = wskazanie po − wskazanie przed (odsyp ujemna, zasyp dodatnia), średnia prędkość = różnica / czas trwania sygnału. Wskazania przed i po liczone dwiema metodami (`METHODS`), do porównania:

- **mediana 3 s** — mediana z 3 s przed początkiem i z 3 s po końcu,
- **najbliższa próbka** — wskazanie najbliższe w czasie początkowi i końcowi sygnału, bez wygładzania.

Kolizja = zdarzenie drugiej roli w odległości mniejszej niż `COLLISION_MARGIN`. Przedziały histogramów różnic to wielokrotności działki (20 kg).

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from dms_analysis.event_stats import (
    METHODS,
    differences_by_method,
    event_differences,
    histogram,
    method_histograms,
)
from dms_analysis.prepared import load_session
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.snapshot import ANALYSIS_SESSION, ANALYSIS_SOURCE
from dms_analysis.tables import collapsible

DB_PATH = Path("../data/prepared/obiekt-sesja-9.db")
FIGURE_WIDTH = 1200
DIFFERENCE_BIN_KG = 20
DURATION_BIN_S = 2
RATE_BIN_KG_S = 0.5

pd.set_option("display.max_rows", 500)

In [ ]:
session = load_session(DB_PATH, ANALYSIS_SOURCE, ANALYSIS_SESSION, CURRENT_VARIANT)
table = event_differences(session)
events = table[table["method"] == next(iter(METHODS))]
rejected = session.events[session.events["rejected"]].groupby("role").size().rename("odrzucone")

for name, method in METHODS.items():
    print(f"{name}: {method.description()}")
events.groupby("role").agg(przyjęte=("start", "size"), kolizje=("collision", "sum")).join(rejected)

In [ ]:
table.groupby(["role", "method"], sort=False)[["difference_kg", "rate_kg_s"]].describe().round(1)

## Odsypy

Prędkość odsypu policzona dodatkowo (przyda się przy wyliczaniu mas przy kolizjach). Czas trwania nie zależy od metody.

In [ ]:
dispense = table[(table["role"] == "odsyp") & ~table["collision"]]

display(histogram(events.loc[events["role"] == "odsyp", "duration_s"], "Odsyp: czas trwania", "czas trwania [s]", DURATION_BIN_S, FIGURE_WIDTH))
display(method_histograms(dispense, "difference_kg", "Odsyp: różnica wskazań", "różnica [kg]", DIFFERENCE_BIN_KG, FIGURE_WIDTH))
display(method_histograms(dispense, "rate_kg_s", "Odsyp: średnia prędkość", "prędkość [kg/s]", RATE_BIN_KG_S, FIGURE_WIDTH))

Różnice metod dla każdego odsypu, posortowane po największej rozbieżności między metodami (rozwiń):

In [ ]:
comparison = differences_by_method(dispense)
spread = comparison[list(METHODS)].max(axis=1) - comparison[list(METHODS)].min(axis=1)
collapsible(comparison.assign(rozrzut_kg=spread).sort_values("rozrzut_kg", ascending=False), "Różnice metod dla odsypów")

Odsypy z różnicą nieujemną w którejkolwiek metodzie (wbrew oczekiwanemu spadkowi):

In [ ]:
comparison[(comparison[list(METHODS)] >= 0).any(axis=1)]

## Zasypy

Zasypów jest kilka, więc zamiast histogramu tabela; histogram ma sens dopiero na większej liczbie danych.

In [ ]:
collapsible(table[table["role"] == "zasyp"].sort_values("start", kind="stable"), "Zasypy")